# **Phenotypic Bias in Heart Disease Prediction**
### *Fairness and Explainability Analysis using the UCI Heart Disease Dataset (Cleveland)*

**Author:** Francisca Mihalache  
**Course:** Artificial Intelligence and Society — Individual Assignment  
**Instructor:** Miriam Seoane Santos  


### **Objective**
This notebook explores potential *phenotypic bias* in heart disease prediction models by analyzing whether performance differs across chest pain types (`cp`) — *typical angina*, *atypical angina*, *non-anginal pain*, and *asymptomatic* — in the **UCI Heart Disease (Cleveland)** dataset.  

The study integrates two key Responsible AI dimensions:  
- **Fairness:** evaluating performance disparities between clinical subgroups.  
- **Explainability:** using SHAP values to understand which features contribute to those differences.  

Results are discussed from both a **technical** and **ethical** standpoint, reflecting on possible subdiagnosis risks for patients with less typical symptoms.



### **Notebook Sections**
1. Data loading and exploration  
2. Preprocessing and feature scaling  
3. Model training (Logistic Regression and Random Forest)  
4. Evaluation and fairness analysis by chest pain type  
5. Explainability with SHAP  
6. Discussion of results and ethical reflection  
7. (Optional) Mitigation and future work


#### Environment Setup


In [1]:
pip install ucimlrepo

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


#### Imports

In [ ]:
from ucimlrepo import fetch_ucirepo 


## 1- Data loading and exploration

#### Dataset Loading and Description
We fetch the UCI Heart Disease (Cleveland) dataset using the ucimlrepo package.
The dataset includes 303 patients and 13 clinical features related to cardiovascular risk.

In [11]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

# Fetch dataset
heart_disease = fetch_ucirepo(id=45)

# Data as pandas DataFrames
X = heart_disease.data.features
y = heart_disease.data.targets

# Display concise metadata
meta_df = pd.DataFrame(list(heart_disease.metadata.items()), columns=["Key", "Value"])
display(meta_df)  

# Display only first 10 variable descriptions
display(heart_disease.variables)

,Key,Value
0,uci_id,45
1,name,Heart Disease
2,repository_url,https://archive.ics.uci.edu/dataset/45/heart+d...
3,data_url,https://archive.ics.uci.edu/static/public/45/d...
4,abstract,"4 databases: Cleveland, Hungary, Switzerland, ..."
5,area,Health and Medicine
6,tasks,[Classification]
7,characteristics,[Multivariate]
8,num_instances,303
9,num_features,13


,name,role,type,demographic,description,units,missing_values
0,age,Feature,Integer,Age,None,years,no
1,sex,Feature,Categorical,Sex,None,None,no
2,cp,Feature,Categorical,None,None,None,no
3,trestbps,Feature,Integer,None,resting blood pressure (on admission to the ho...,mm Hg,no
4,chol,Feature,Integer,None,serum cholestoral,mg/dl,no
5,fbs,Feature,Categorical,None,fasting blood sugar > 120 mg/dl,None,no
6,restecg,Feature,Categorical,None,None,None,no
7,thalach,Feature,Integer,None,maximum heart rate achieved,None,no
8,exang,Feature,Categorical,None,exercise induced angina,None,no
9,oldpeak,Feature,Integer,None,ST depression induced by exercise relative to ...,None,no


The tables above summarize the **dataset metadata** and the **full list of variables** retrieved from the UCI Machine Learning Repository.  
The original code example from the `ucimlrepo` documentation was slightly modified to present the information in a more compact and structured way.

From the **metadata table**, we confirm that:

- **Task type:** Classification (predicting presence of heart disease)  
- **Instances:** 303 patients  
- **Features:** 13 clinical attributes (a mix of categorical, integer, and real types)  
- **Demographic variables:** Age and Sex  
- **Target variable:** `num` — ranges from 0 (no disease) to 4 (presence of disease)  
- **Missing values:** Yes — represented as `NaN`  
- **Dataset creation year:** 1989, updated in 2023  
- **Creators:** Andras Janosi, William Steinbrunn, Matthias Pfisterer, Robert Detrano  
- **DOI:** [10.24432/C52P4X](https://doi.org/10.24432/C52P4X)  
- **Repository link:** [UCI Heart Disease Dataset](https://archive.ics.uci.edu/dataset/45/heart+disease)

From the **variable table**, we can summarize:
- There are 13 input features (`age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`, `oldpeak`, `slope`, `ca`, `thal`) and 1 target variable (`num`).  
- Most features are either **integer** or **categorical**, aligning with clinical measurements and diagnostic categories.  
- The `cp` (chest pain type) variable is especially important for this project — it differentiates patients by pain presentation (*typical angina*, *atypical angina*, *non-anginal pain*, *asymptomatic*) and will be used to study *phenotypic bias*.  
- The variables `ca` (number of major vessels) and `thal` contain missing values.  
- Descriptive units include `mm Hg` (for blood pressure) and `mg/dl` (for cholesterol).  

This metadata inspection confirms that the dataset is well-suited for the intended analysis on **fairness** and **explainability**.  

Next, we proceed to **data preprocessing**, where the target variable will be binarized and missing values will be handled before training the classification models.
